# 05b - (Tahap 4) Diversity, Konvergensi, dan Master QC Report

Notebook ini melanjutkan Tahap 2 (rule-based) dan Tahap 3 (LLM-as-judge).

Urutan:
1. Restore hasil Tahap 2 dan Tahap 3
2. Uji konvergensi ukuran sampel (self-BLEU dan embedding diversity)
3. Analisis diversity populasi penuh
4. Master QC report gabungan semua stage

In [1]:
!pip install nltk -q

In [2]:
import os
import csv
import json
import math
import time
import random
import shutil
import platform
import datetime
from collections import defaultdict, Counter
from pathlib import Path

import pandas as pd
import torch
import glob
import nltk
nltk.download("punkt", quiet=True)

True

## 1. Konfigurasi

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE} cuda={torch.cuda.is_available()}")

def load_records(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records
 
def find_newest(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    for p in hits:
        print("  kandidat:", p, "|", os.path.getmtime(p))
    return max(hits, key=os.path.getmtime)
 
 
INPUT_PATH = find_newest("/kaggle/input/**/filtered_records.jsonl")
print("DIPILIH input_path:", INPUT_PATH)
 
CONFIG = {
    "input_path": INPUT_PATH,
    "output_dir": "/kaggle/working/qc",
    "stratify_key": "subtask_type",
    "embedding_model": "firqaaa/Indo-Sentence-BERT-Base",
    "seed": 42,
    "device": DEVICE,
    "embed_batch_size": 32,
    "diversity_selfbleu_fraction": 0.035,
    "diversity_embedding_fraction": 0.13,
    "opener_ngram_n": 3,
    "opener_top_k": 30,
    "convergence_n_repeats": 3,
    "convergence_selfbleu_fractions": [0.01, 0.02, 0.035, 0.05],
    "convergence_embedding_fractions": [0.05, 0.10, 0.15, 0.20],
    "convergence_cv_threshold": 5.0,
    "convergence_delta_threshold": 0.01,
    "test_mode": False,
    "diversity_records_limit_test": 50,
    "diversity_selfbleu_sample_test": 20,
    "diversity_embedding_sample_test": 20,
    "convergence_selfbleu_fractions_test": [0.05, 0.10],
    "convergence_embedding_fractions_test": [0.05, 0.10],
    "convergence_n_repeats_test": 2,
}
 
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
random.seed(CONFIG["seed"])
os.environ["PYTHONHASHSEED"] = str(CONFIG["seed"])
print(f"test_mode={CONFIG['test_mode']}")
 

_recs = load_records(CONFIG["input_path"])
_gate = ["schema_invalid", "chunk_id_invalid", "degenerate",
         "table_ref_instruction", "near_duplicate"]
_rr = sum(any((r.get("flags", {}) or {}).get(k) is True for k in _gate) for r in _recs)
print(f"total records di filtered_records : {len(_recs):,}")
print(f"punya flags rule-based aktif       : {_rr:,}")
print(f"contoh keys record                 : {list(_recs[0].keys())}")
print(f"contoh flags                       : {_recs[0].get('flags')}")

device=cuda cuda=True
  kandidat: /kaggle/input/notebooks/muhammadnabilul/05b-tahap-2-cleaning-dataset-with-rule-based/output_05b/filtered_records.jsonl | 1784817878.4338806
DIPILIH input_path: /kaggle/input/notebooks/muhammadnabilul/05b-tahap-2-cleaning-dataset-with-rule-based/output_05b/filtered_records.jsonl
test_mode=False
total records di filtered_records : 20,855
punya flags rule-based aktif       : 0
contoh keys record                 : ['record_id', 'dataset_version', 'task_type', 'subtask_type', 'generation_focus', 'reasoning_source', 'context_focus', 'instruction', 'response', 'context', 'context_hash', 'context_token_count', 'context_chunk_ids', 'source_pages', 'batch_index', 'num_batches', 'document_title', 'document_year', 'section_id', 'section_title', 'bps_url', 'parse_status', 'prompt_tokens', 'completion_tokens', 'cached_tokens', 'generation_model', 'generation_timestamp', 'flags', 'numeric_score', 'track_a_score']
contoh flags                       : {'unmatched_numbe

### Fungsi umum

In [4]:
def write_csv(rows, path, fieldnames):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)


def read_csv(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_summary(summary, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)


def parse_bool(value):
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    return str(value).strip().lower() in ("true", "1", "yes")


def preview_rows(rows, sort_key, n=5, reverse=False, label=""):
    def key_fn(r):
        v = r.get(sort_key)
        try:
            return float(v)
        except (TypeError, ValueError):
            return -1.0
    ordered = sorted(rows, key=key_fn, reverse=reverse)
    print(f"\n--- preview {label} ({'top' if reverse else 'bottom'} {n} by {sort_key}) ---")
    for r in ordered[:n]:
        print(f"  {r.get('record_id', r.get('subtask_type', r.get('opener', '?')))} | "
              f"{sort_key}={r.get(sort_key)}")

## 2. Restore dari Tahap sebelumnya

In [5]:
def restore_from_previous_stages(config):
    working_dir = Path(config["output_dir"])
    working_dir.mkdir(parents=True, exist_ok=True)

    files_to_restore = [
        "stage1_prefilter.csv",
        "stage1_summary.json",
        "stage2_ragas.csv",
        "stage2_ragas_checkpoint.csv",
        "stage2c_deepeval.csv",
        "stage2c_deepeval_checkpoint.csv",
        "stage3_human_eval_template.csv",
        "stage3_summary.json",
        "environment_snapshot.json",
        "stage4_self_bleu_convergence.csv",
        "stage4_embedding_diversity_convergence.csv",
        "stage4_diversity_summary.csv",
        "stage4_stratum_coverage.csv",
        "stage4_opener_distribution.csv",
        "stage4_summary.json",
    ]

    kaggle_input = Path("/kaggle/input")
    restored, missing = [], []

    for fname in files_to_restore:
        target = working_dir / fname
        if target.exists() and target.stat().st_size > 0:
            continue
        candidates = ([p for p in kaggle_input.rglob(fname)
                       if p.is_file() and p.stat().st_size > 0]
                      if kaggle_input.exists() else [])
        if not candidates:
            missing.append(fname)
            continue
        source = max(candidates, key=lambda p: p.stat().st_mtime)
        shutil.copy2(source, target)
        restored.append(f"{fname} <- {source.parent}")

    if restored:
        print("dipulihkan dari input:")
        for r in restored:
            print(f"  {r}")
    if missing:
        print("\nbelum tersedia (wajar bila stage terkait belum dijalankan):")
        for m in missing:
            print(f"  {m}")

    print("\nisi working dir:")
    for f in sorted(working_dir.iterdir()):
        if f.is_file():
            print(f"  {f.name}  {f.stat().st_size:,} bytes")

    return restored


restore_from_previous_stages(CONFIG)

dipulihkan dari input:
  stage1_prefilter.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  stage1_summary.json <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  stage2_ragas.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  stage2_ragas_checkpoint.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  stage2c_deepeval.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  stage2c_deepeval_checkpoint.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  stage3_human_eval_template.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  stage3_summary.json <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc
  environment_snapshot.json <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-

['stage1_prefilter.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'stage1_summary.json <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'stage2_ragas.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'stage2_ragas_checkpoint.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'stage2c_deepeval.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'stage2c_deepeval_checkpoint.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'stage3_human_eval_template.csv <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'stage3_summary.json <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleaning-dataset-with-ragas/qc',
 'environment_snapshot.json <- /kaggle/input/notebooks/muhammadnabilul/05b-tahap-3-cleanin

### Log ENV

In [6]:
def log_environment(config):
    import importlib.metadata as md

    packages = ["nltk", "sentence-transformers", "torch", "pandas", "numpy"]
    versions = {}
    for p in packages:
        try:
            versions[p] = md.version(p)
        except Exception:
            versions[p] = "not installed"

    env = {
        "stage": "tahap_4_diversity_master_report",
        "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "device": config["device"],
        "cuda_available": torch.cuda.is_available(),
        "seed": config["seed"],
        "packages": versions,
        "diversity_params": {
            "selfbleu_fraction": config["diversity_selfbleu_fraction"],
            "embedding_fraction": config["diversity_embedding_fraction"],
            "opener_ngram_n": config["opener_ngram_n"],
            "embedding_model": config["embedding_model"],
        },
    }

    path = f"{config['output_dir']}/environment_snapshot_tahap4.json"
    write_summary(env, path)
    print("environment tahap 4:")
    for k, v in versions.items():
        print(f"  {k}: {v}")
    print(f"disimpan ke: {path}")
    return env


env_tahap4 = log_environment(CONFIG)

environment tahap 4:
  nltk: 3.9.1
  sentence-transformers: 5.4.1
  torch: 2.10.0+cu128
  pandas: 2.3.3
  numpy: 2.0.2
disimpan ke: /kaggle/working/qc/environment_snapshot_tahap4.json


## 3. Diversity test

### Fungsi Metrik Diversity

Lima kelompok metrik, masing-masing mengukur sudut keragaman yang berbeda:

| Fungsi | Yang diukur | Arah baik |
|---|---|---|
| `distinct_n` | keragaman leksikal (rasio n-gram unik) | tinggi |
| `self_bleu` | kemiripan antar-pertanyaan | **rendah** |
| `embedding_diversity` | kemiripan semantik (mean pairwise cosine) | **rendah** |
| `opener_distribution` | konsentrasi pola pembuka kalimat | tersebar |
| `stratum_coverage` | keseimbangan antar-`subtask_type` | entropy tinggi, gini rendah |

`self_bleu` dan `embedding_diversity` berbiaya kuadratik terhadap jumlah sampel,
sehingga keduanya dihitung atas sub-sampel — inilah yang diuji konvergensinya di blok berikutnya.

In [7]:
def tokenize(text):
    return text.lower().split()


def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def distinct_n(questions, n):
    total = 0
    unique = set()
    for q in questions:
        grams = ngrams(tokenize(q), n)
        total += len(grams)
        unique.update(grams)
    return len(unique) / total if total else 0.0


def self_bleu(questions, sample_size, seed):
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    rng = random.Random(seed)
    sample = rng.sample(questions, min(sample_size, len(questions)))
    toks = [tokenize(q) for q in sample]
    sm = SmoothingFunction().method1
    scores = []
    for i, hyp in enumerate(toks):
        refs = toks[:i] + toks[i + 1:]
        if refs and hyp:
            scores.append(sentence_bleu(refs, hyp, smoothing_function=sm))
    return sum(scores) / len(scores) if scores else 0.0


def embedding_diversity(questions, model_name, sample_size, seed, device, batch_size):
    from sentence_transformers import SentenceTransformer, util
    rng = random.Random(seed)
    sample = rng.sample(questions, min(sample_size, len(questions)))
    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(sample, convert_to_tensor=True, batch_size=batch_size,
                       normalize_embeddings=True, show_progress_bar=False)
    sim = util.cos_sim(emb, emb)
    n = sim.shape[0]
    return (sim.sum().item() - n) / (n * (n - 1)) if n > 1 else 0.0


def opener_distribution(questions, n, top_k):
    counter = Counter()
    for q in questions:
        toks = tokenize(q)[:n]
        if toks:
            counter[" ".join(toks)] += 1
    total = sum(counter.values())
    rows = [{"opener": k, "count": c, "proportion": round(c / total, 4)}
            for k, c in counter.most_common(top_k)]
    return rows, total


def stratum_coverage(records, stratify_key):
    by = defaultdict(list)
    for r in records:
        by[r.get(stratify_key, "?")].append(r.get("instruction", ""))

    total = sum(len(v) for v in by.values())
    k = len(by)
    props = [len(v) / total for v in by.values()]
    entropy = -sum(p * math.log(p) for p in props if p > 0)
    norm_entropy = entropy / math.log(k) if k > 1 else 0.0

    counts_sorted = sorted(len(v) for v in by.values())
    n = len(counts_sorted)
    gini_num = sum((2 * i - n - 1) * c for i, c in enumerate(counts_sorted, 1))
    gini = gini_num / (n * total) if total else 0.0

    rows = []
    for s in sorted(by):
        qs = by[s]
        rows.append({
            "subtask_type": s,
            "count": len(qs),
            "proportion": round(len(qs) / total, 4),
            "distinct_2": round(distinct_n(qs, 2), 4),
        })
    return rows, norm_entropy, gini

## 4. Konvergensi ukuran sample

### Uji Konvergensi Ukuran Sampel

**Tujuan:** menentukan fraksi sampel terkecil yang sudah menghasilkan estimasi stabil, sehingga pemilihan ukuran sampel punya dasar empiris — bukan angka yang dipilih sembarang.

**Cara kerja:** metrik dihitung berulang kali pada fraksi berbeda dengan seed berbeda. Jika hasil antar-seed sudah rapat (koefisien variasi kecil) dan penambahan fraksi tidak lagi menggeser rata-rata, fraksi tersebut dianggap konvergen.

**Kriteria yang dipakai:** `cv% < 5` dan `Δmean < 0.01` terhadap fraksi sebelumnya.


In [8]:
def convergence_test_by_fraction(config, metric, fractions, n_repeats):
    records = load_records(config["input_path"])
    questions = [r.get("instruction", "") for r in records if r.get("instruction", "").strip()]
    N = len(questions)
    print(f"populasi pertanyaan: {N:,}")
    print(f"metrik: {metric} | fraksi diuji: {fractions} | repeat: {n_repeats}\n")

    rows = []
    for frac in fractions:
        n = max(30, int(round(N * frac)))
        if n > N:
            continue

        vals, durations = [], []
        for rep in range(n_repeats):
            seed = config["seed"] + rep * 1000
            t0 = time.time()
            if metric == "self_bleu":
                score = self_bleu(questions, n, seed)
            else:
                score = embedding_diversity(questions, config["embedding_model"], n,
                                            seed, config["device"], config["embed_batch_size"])
            elapsed = time.time() - t0
            vals.append(score)
            durations.append(elapsed)
            print(f"  frac={frac:.3f} n={n:5d} rep={rep+1}/{n_repeats} "
                  f"seed={seed} {metric}={score:.4f} ({elapsed:.1f}s)")

        mean = sum(vals) / len(vals)
        std = (sum((v - mean) ** 2 for v in vals) / len(vals)) ** 0.5
        rows.append({
            "fraction": frac,
            "sample_size": n,
            "population": N,
            f"mean_{metric}": round(mean, 4),
            f"std_{metric}": round(std, 5),
            "cv_percent": round(std / mean * 100, 2) if mean else 0.0,
            "min": round(min(vals), 4),
            "max": round(max(vals), 4),
            "range": round(max(vals) - min(vals), 4),
            "mean_seconds": round(sum(durations) / len(durations), 1),
            "n_repeats": n_repeats,
        })
        print(f"  -> frac={frac:.3f}: mean={mean:.4f} std={std:.5f} "
              f"cv={rows[-1]['cv_percent']:.2f}%\n")

    out_path = f"{config['output_dir']}/stage4_{metric}_convergence.csv"
    write_csv(rows, out_path, list(rows[0].keys()))

    print(f"--- ringkasan konvergensi: {metric} ---")
    print(f"{'frac':>7} {'n':>7} {'mean':>9} {'std':>9} {'cv%':>7} {'detik':>9}  {'d_mean':>8}")
    prev = None
    for r in rows:
        d = "-" if prev is None else f"{abs(r[f'mean_{metric}'] - prev):.4f}"
        print(f"{r['fraction']:>7.3f} {r['sample_size']:>7,} {r[f'mean_{metric}']:>9.4f} "
              f"{r[f'std_{metric}']:>9.5f} {r['cv_percent']:>6.2f}% "
              f"{r['mean_seconds']:>9.1f}  {d:>8}")
        prev = r[f"mean_{metric}"]

    print(f"\ndisimpan ke: {out_path}")
    print("cara baca: pilih fraksi terkecil dengan cv% di bawah ambang "
          "dan d_mean sudah kecil terhadap fraksi sebelumnya.")
    return rows


def run_convergence_if_needed(config, metric, fractions, n_repeats):
    path = f"{config['output_dir']}/stage4_{metric}_convergence.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        rows = read_csv(path)
        print(f"konvergensi {metric}: SKIP, sudah ada ({len(rows)} baris di {path})")
        print("        hapus file tersebut bila ingin menghitung ulang.")
        return rows
    return convergence_test_by_fraction(config, metric, fractions, n_repeats)

### 4a. Konvergensi Self-BLEU

Fraksi yang diuji sengaja kecil (1–5%). Self-BLEU membandingkan setiap kalimat dengan
seluruh kalimat lain, sehingga jumlah operasinya tumbuh kuadratik: 5% dari 15.000
pertanyaan sudah berarti ratusan ribu perbandingan BLEU.

In [9]:
_fracs = (CONFIG["convergence_selfbleu_fractions_test"] if CONFIG["test_mode"]
          else CONFIG["convergence_selfbleu_fractions"])
_reps = (CONFIG["convergence_n_repeats_test"] if CONFIG["test_mode"]
         else CONFIG["convergence_n_repeats"])

selfbleu_convergence = run_convergence_if_needed(CONFIG, "self_bleu", _fracs, _reps)

populasi pertanyaan: 20,855
metrik: self_bleu | fraksi diuji: [0.01, 0.02, 0.035, 0.05] | repeat: 3

  frac=0.010 n=  209 rep=1/3 seed=42 self_bleu=0.2674 (1.8s)
  frac=0.010 n=  209 rep=2/3 seed=1042 self_bleu=0.2955 (1.8s)
  frac=0.010 n=  209 rep=3/3 seed=2042 self_bleu=0.3081 (1.8s)
  -> frac=0.010: mean=0.2903 std=0.01701 cv=5.86%

  frac=0.020 n=  417 rep=1/3 seed=42 self_bleu=0.3552 (7.1s)
  frac=0.020 n=  417 rep=2/3 seed=1042 self_bleu=0.3587 (7.3s)
  frac=0.020 n=  417 rep=3/3 seed=2042 self_bleu=0.3810 (7.3s)
  -> frac=0.020: mean=0.3650 std=0.01143 cv=3.13%

  frac=0.035 n=  730 rep=1/3 seed=42 self_bleu=0.4170 (22.1s)
  frac=0.035 n=  730 rep=2/3 seed=1042 self_bleu=0.4150 (22.4s)
  frac=0.035 n=  730 rep=3/3 seed=2042 self_bleu=0.4403 (22.5s)
  -> frac=0.035: mean=0.4241 std=0.01148 cv=2.71%

  frac=0.050 n= 1043 rep=1/3 seed=42 self_bleu=0.4629 (44.7s)
  frac=0.050 n= 1043 rep=2/3 seed=1042 self_bleu=0.4539 (45.5s)
  frac=0.050 n= 1043 rep=3/3 seed=2042 self_bleu=0.4742 

### 4b. Konvergensi Embedding Diversity

Fraksi yang diuji lebih besar (5–20%) karena perhitungan matriks cosine jauh lebih
ringan daripada BLEU, meski jumlah pasangannya sama-sama kuadratik.

In [10]:
_fracs = (CONFIG["convergence_embedding_fractions_test"] if CONFIG["test_mode"]
          else CONFIG["convergence_embedding_fractions"])
_reps = (CONFIG["convergence_n_repeats_test"] if CONFIG["test_mode"]
         else CONFIG["convergence_n_repeats"])

embedding_convergence = run_convergence_if_needed(CONFIG, "embedding_diversity", _fracs, _reps)

populasi pertanyaan: 20,855
metrik: embedding_diversity | fraksi diuji: [0.05, 0.1, 0.15, 0.2] | repeat: 3



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: firqaaa/Indo-Sentence-BERT-Base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  frac=0.200 n= 4171 rep=2/3 seed=1042 embedding_diversity=0.3091 (12.3s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: firqaaa/Indo-Sentence-BERT-Base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  frac=0.200 n= 4171 rep=3/3 seed=2042 embedding_diversity=0.3115 (12.4s)
  -> frac=0.200: mean=0.3097 std=0.00126 cv=0.41%

--- ringkasan konvergensi: embedding_diversity ---
   frac       n      mean       std     cv%     detik    d_mean
  0.050   1,043    0.3073   0.00198   0.64%      17.2         -
  0.100   2,086    0.3077   0.00281   0.91%       7.8    0.0004
  0.150   3,128    0.3085   0.00295   0.96%       9.9    0.0008
  0.200   4,171    0.3097   0.00126   0.41%      12.2    0.0012

disimpan ke: /kaggle/working/qc/stage4_embedding_diversity_convergence.csv
cara baca: pilih fraksi terkecil dengan cv% di bawah ambang dan d_mean sudah kecil terhadap fraksi sebelumnya.


### 4c. Rekomendasi Fraksi

Sel ini hanya **membaca** tabel konvergensi dan mencetak fraksi terkecil yang memenuhi
kriteria. Nilai di CONFIG tidak diubah otomatis — keputusan tetap di tangan peneliti dan
sebaiknya dituliskan manual di sel Konfigurasi sebelum commit berikutnya.

In [11]:
def recommend_fraction(config, metric):
    path = f"{config['output_dir']}/stage4_{metric}_convergence.csv"
    if not os.path.exists(path):
        print(f"{metric}: tabel konvergensi belum ada.")
        return None

    rows = sorted(read_csv(path), key=lambda r: float(r["fraction"]))
    mean_col = f"mean_{metric}"
    cv_thr = config["convergence_cv_threshold"]
    d_thr = config["convergence_delta_threshold"]

    prev, chosen = None, None
    for r in rows:
        cv = float(r["cv_percent"])
        mean = float(r[mean_col])
        delta = abs(mean - prev) if prev is not None else None
        ok_cv = cv < cv_thr
        ok_delta = delta is not None and delta < d_thr
        mark = "OK" if (ok_cv and ok_delta) else "  "
        d_str = "-" if delta is None else f"{delta:.4f}"
        print(f"  {mark} frac={float(r['fraction']):.3f} n={int(r['sample_size']):>6,} "
              f"mean={mean:.4f} cv={cv:.2f}% d_mean={d_str}")
        if chosen is None and ok_cv and ok_delta:
            chosen = float(r["fraction"])
        prev = mean

    print(f"\n  kriteria: cv% < {cv_thr}, d_mean < {d_thr}")
    if chosen is not None:
        print(f"  -> rekomendasi fraksi {metric}: {chosen}")
    else:
        largest = float(rows[-1]["fraction"])
        print(f"  -> tidak ada fraksi yang memenuhi kedua kriteria. "
              f"Pertimbangkan fraksi lebih besar dari {largest}, "
              f"atau naikkan n_repeats agar estimasi std lebih andal.")
    return chosen


print("=== self_bleu ===")
rec_selfbleu = recommend_fraction(CONFIG, "self_bleu")
print("\n=== embedding_diversity ===")
rec_embedding = recommend_fraction(CONFIG, "embedding_diversity")

print(f"\nfraksi yang sedang dipakai CONFIG:")
print(f"  diversity_selfbleu_fraction  = {CONFIG['diversity_selfbleu_fraction']}")
print(f"  diversity_embedding_fraction = {CONFIG['diversity_embedding_fraction']}")

=== self_bleu ===
     frac=0.010 n=   209 mean=0.2903 cv=5.86% d_mean=-
     frac=0.020 n=   417 mean=0.3650 cv=3.13% d_mean=0.0747
     frac=0.035 n=   730 mean=0.4241 cv=2.71% d_mean=0.0591
     frac=0.050 n= 1,043 mean=0.4637 cv=1.80% d_mean=0.0396

  kriteria: cv% < 5.0, d_mean < 0.01
  -> tidak ada fraksi yang memenuhi kedua kriteria. Pertimbangkan fraksi lebih besar dari 0.05, atau naikkan n_repeats agar estimasi std lebih andal.

=== embedding_diversity ===
     frac=0.050 n= 1,043 mean=0.3073 cv=0.64% d_mean=-
  OK frac=0.100 n= 2,086 mean=0.3077 cv=0.91% d_mean=0.0004
  OK frac=0.150 n= 3,128 mean=0.3085 cv=0.96% d_mean=0.0008
  OK frac=0.200 n= 4,171 mean=0.3097 cv=0.41% d_mean=0.0012

  kriteria: cv% < 5.0, d_mean < 0.01
  -> rekomendasi fraksi embedding_diversity: 0.1

fraksi yang sedang dipakai CONFIG:
  diversity_selfbleu_fraction  = 0.035
  diversity_embedding_fraction = 0.13


## 5. Analisis Diversity Populasi

Dihitung atas **seluruh** record yang lolos Tahap 2, bukan atas sampel yang dinilai LLM-judge — keragaman adalah properti populasi dataset, bukan properti sampel evaluasi.

`distinct_n`, `opener_distribution`, dan `stratum_coverage` menggunakan populasi penuh.
Hanya `self_bleu` dan `mean_pairwise_cosine` yang memakai sub-sampel; ukuran sampel yang
dipakai ikut dicatat pada file keluaran agar angka yang dilaporkan dapat ditelusuri.

In [12]:
def stage4_diversity_coverage(config):
    records = load_records(config["input_path"])
    suffix = "_test" if config.get("test_mode") else ""

    if config.get("test_mode") and config.get("diversity_records_limit_test"):
        limit = config["diversity_records_limit_test"]
        records = records[:limit]
        print(f"[TEST MODE] diversity dihitung dari {len(records)} record pertama saja")

    questions = [r.get("instruction", "") for r in records if r.get("instruction", "").strip()]
    n_q = len(questions)

    if config.get("test_mode"):
        selfbleu_n = config["diversity_selfbleu_sample_test"]
        embed_n = config["diversity_embedding_sample_test"]
    else:
        selfbleu_n = max(30, int(round(n_q * config["diversity_selfbleu_fraction"])))
        embed_n = max(30, int(round(n_q * config["diversity_embedding_fraction"])))

    print(f"populasi pertanyaan : {n_q:,}")
    print(f"sampel self_bleu    : {selfbleu_n:,} ({selfbleu_n/n_q*100:.2f}%)")
    print(f"sampel embedding    : {embed_n:,} ({embed_n/n_q*100:.2f}%)")

    print("\nmenghitung distinct_n ...")
    d1 = distinct_n(questions, 1)
    d2 = distinct_n(questions, 2)

    print("menghitung self_bleu ...")
    t0 = time.time()
    sb = self_bleu(questions, selfbleu_n, config["seed"])
    print(f"  selesai dalam {time.time()-t0:.1f}s")

    print("menghitung mean_pairwise_cosine ...")
    t0 = time.time()
    mpc = embedding_diversity(questions, config["embedding_model"], embed_n,
                              config["seed"], config["device"], config["embed_batch_size"])
    print(f"  selesai dalam {time.time()-t0:.1f}s")

    summary = [{
        "n_questions": n_q,
        "distinct_1": round(d1, 4),
        "distinct_2": round(d2, 4),
        "self_bleu": round(sb, 4),
        "self_bleu_sample_size": selfbleu_n,
        "self_bleu_fraction": round(selfbleu_n / n_q, 4),
        "mean_pairwise_cosine": round(mpc, 4),
        "embedding_sample_size": embed_n,
        "embedding_fraction": round(embed_n / n_q, 4),
        "embedding_model": config["embedding_model"],
    }]
    write_csv(summary, f"{config['output_dir']}/stage4_diversity_summary{suffix}.csv",
              list(summary[0].keys()))

    print("\n--- diversity summary ---")
    for k, v in summary[0].items():
        print(f"  {k}: {v}")
    print("  (distinct_n: makin tinggi makin beragam | "
          "self_bleu & mean_pairwise_cosine: makin RENDAH makin beragam)")

    strat_rows, norm_entropy, gini = stratum_coverage(records, config["stratify_key"])
    write_csv(strat_rows, f"{config['output_dir']}/stage4_stratum_coverage{suffix}.csv",
              ["subtask_type", "count", "proportion", "distinct_2"])
    print(f"\nstratum coverage: norm_entropy={norm_entropy:.3f} gini={gini:.3f}")
    print("  (norm_entropy mendekati 1 = stratum seimbang | gini mendekati 0 = seimbang)")
    print("  CATATAN: distinct_2 per stratum dipengaruhi ukuran stratum — "
          "stratum kecil cenderung terlihat lebih beragam. Bandingkan bersama kolom count.")
    preview_rows(strat_rows, "count", n=5, reverse=False, label="stratum terkecil")
    preview_rows(strat_rows, "count", n=5, reverse=True, label="stratum terbesar")

    opener_rows, opener_total = opener_distribution(
        questions, config["opener_ngram_n"], config["opener_top_k"])
    write_csv(opener_rows, f"{config['output_dir']}/stage4_opener_distribution{suffix}.csv",
              ["opener", "count", "proportion"])
    top5 = sum(r["proportion"] for r in opener_rows[:5]) if opener_rows else 0.0
    print(f"\nkonsentrasi pembuka (n={config['opener_ngram_n']} kata): "
          f"top-5 mencakup {top5*100:.1f}% dari {opener_total:,} pertanyaan")
    for r in opener_rows[:10]:
        print(f"  {r['count']:>6,} ({r['proportion']*100:>5.2f}%)  {r['opener']}")

    write_summary({
        "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "n_questions": n_q,
        "distinct_1": round(d1, 4),
        "distinct_2": round(d2, 4),
        "self_bleu": round(sb, 4),
        "mean_pairwise_cosine": round(mpc, 4),
        "norm_entropy": round(norm_entropy, 4),
        "gini": round(gini, 4),
        "top5_opener_share": round(top5, 4),
        "sample_sizes": {"self_bleu": selfbleu_n, "embedding": embed_n},
        "fractions_used": {
            "self_bleu": config["diversity_selfbleu_fraction"],
            "embedding": config["diversity_embedding_fraction"],
        },
        "test_mode": config.get("test_mode", False),
    }, f"{config['output_dir']}/stage4_summary{suffix}.json")

    return {"summary": summary, "strata": strat_rows, "openers": opener_rows,
            "norm_entropy": norm_entropy, "gini": gini}


def run_stage4_if_needed(config):
    suffix = "_test" if config.get("test_mode") else ""
    path = f"{config['output_dir']}/stage4_stratum_coverage{suffix}.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"stage4 diversity: SKIP, sudah ada {path}")
        print("        hapus file tersebut bila ingin menghitung ulang.")
        return None
    return stage4_diversity_coverage(config)


diversity_report = run_stage4_if_needed(CONFIG)

populasi pertanyaan : 20,855
sampel self_bleu    : 730 (3.50%)
sampel embedding    : 2,711 (13.00%)

menghitung distinct_n ...
menghitung self_bleu ...
  selesai dalam 22.4s
menghitung mean_pairwise_cosine ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: firqaaa/Indo-Sentence-BERT-Base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  selesai dalam 9.6s

--- diversity summary ---
  n_questions: 20855
  distinct_1: 0.0311
  distinct_2: 0.1567
  self_bleu: 0.417
  self_bleu_sample_size: 730
  self_bleu_fraction: 0.035
  mean_pairwise_cosine: 0.3122
  embedding_sample_size: 2711
  embedding_fraction: 0.13
  embedding_model: firqaaa/Indo-Sentence-BERT-Base
  (distinct_n: makin tinggi makin beragam | self_bleu & mean_pairwise_cosine: makin RENDAH makin beragam)

stratum coverage: norm_entropy=0.995 gini=0.074
  (norm_entropy mendekati 1 = stratum seimbang | gini mendekati 0 = seimbang)
  CATATAN: distinct_2 per stratum dipengaruhi ukuran stratum — stratum kecil cenderung terlihat lebih beragam. Bandingkan bersama kolom count.

--- preview stratum terkecil (bottom 5 by count) ---
  document_overview | count=1560
  definition_clarification | count=2085
  methodology_explanation | count=2211
  source_navigation | count=2215
  comparative_analysis | count=2468

--- preview stratum terbesar (top 5 by count) ---
  factual_re

## 6. Master QC Report

Menggabungkan seluruh stage menjadi satu CSV populasi penuh.

**Prinsip `final_decision`:**

1. **Rule-based hard gate** (Tahap 2) — penentu tolak mutlak, sudah final.
2. **IndoBERT prefilter** (Stage 1) — hanya penentu prioritas sampling; tidak pernah
   dipakai untuk menolak atau meloloskan record.
3. **Ragas + DeepEval** (Stage 2/2c) — basis keputusan lolos/tolak, tetapi hanya berlaku
   bagi record yang benar-benar disampel.
4. Record yang lolos rule-based namun tidak disampel LLM-judge berstatus
   `UNVERIFIED_BY_LLM_JUDGE` — bukan `PASSED`, bukan `REJECTED`.
5. **Diversity** (Stage 4) adalah statistik level-populasi per stratum; ditulis pada kolom
   dengan nama eksplisit dan tidak ikut menentukan keputusan per-record.

`high_ocr_noise` dan `mixed_english_detected` diperlakukan sebagai **soft flag**, konsisten
dengan Tahap 2 — dicatat pada kolom tersendiri tanpa mengubah `final_decision`, agar
kriteria penolakan di laporan akhir tidak berbeda dari pipeline hulu.

In [13]:
RULE_BASED_HARD_GATE_KEYS = [
    "schema_invalid",
    "chunk_id_invalid",
    "degenerate",
    "table_ref_instruction",
    "near_duplicate",
]

SOFT_FLAG_KEYS = [
    "high_ocr_noise",
    "mixed_english_detected",
]

LLM_JUDGE_FAITHFULNESS_THRESHOLD = 0.5


def determine_final_decision(rule_based_rejected, is_llm_judged, ragas_f, deepeval_f):
    if rule_based_rejected:
        return "REJECTED_RULE_BASED"
    if not is_llm_judged:
        return "UNVERIFIED_BY_LLM_JUDGE"
    scores = [s for s in (ragas_f, deepeval_f) if s is not None]
    if not scores:
        return "UNVERIFIED_BY_LLM_JUDGE"
    mean_score = sum(scores) / len(scores)
    return "PASSED" if mean_score >= LLM_JUDGE_FAITHFULNESS_THRESHOLD else "REJECTED_LOW_FAITHFULNESS"


raw_records = load_records(CONFIG["input_path"])
print(f"raw_records dimuat: {len(raw_records):,} records | "
      f"mode={'TEST' if CONFIG.get('test_mode') else 'FULL'}")
print(f"hard gate keys : {RULE_BASED_HARD_GATE_KEYS}")
print(f"soft flag keys : {SOFT_FLAG_KEYS}")
print(f"threshold LLM-judge faithfulness: {LLM_JUDGE_FAITHFULNESS_THRESHOLD}")
print("\nCATATAN: threshold 0.5 masih nilai sementara. Idealnya dikalibrasi "
      "dari label human eval (Stage 3) menggunakan ROC / Youden's J.")

raw_records dimuat: 20,855 records | mode=FULL
hard gate keys : ['schema_invalid', 'chunk_id_invalid', 'degenerate', 'table_ref_instruction', 'near_duplicate']
soft flag keys : ['high_ocr_noise', 'mixed_english_detected']
threshold LLM-judge faithfulness: 0.5

CATATAN: threshold 0.5 masih nilai sementara. Idealnya dikalibrasi dari label human eval (Stage 3) menggunakan ROC / Youden's J.


In [14]:
def _to_float(value):
    if value in (None, "", "None", "-"):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def generate_master_csv_report(config, raw_records):
    suffix = "_test" if config.get("test_mode") else ""
    out_dir = config["output_dir"]

    def load_optional(fname, key="record_id"):
        path = f"{out_dir}/{fname}"
        if not os.path.exists(path) or os.path.getsize(path) == 0:
            print(f"  {fname}: TIDAK ADA")
            return {}
        rows = read_csv(path)
        print(f"  {fname}: {len(rows):,} baris")
        return {r[key]: r for r in rows}

    print("memuat artefak stage:")
    stage1_map = load_optional("stage1_prefilter.csv")
    stage2_map = load_optional(f"stage2_ragas{suffix}.csv")
    stage2c_map = load_optional(f"stage2c_deepeval{suffix}.csv")
    stage3_map = load_optional(f"stage3_human_eval_template{suffix}.csv")
    stage4_map = load_optional(f"stage4_stratum_coverage{suffix}.csv", key="subtask_type")

    stage3_ids = set(stage3_map.keys())

    if not stage4_map:
        print("\n[PERINGATAN] artefak stage4 belum tersedia. Kolom stratum_level_* akan "
              "berisi '-'. Jalankan blok Diversity terlebih dahulu bila kolom itu diperlukan.")

    if config.get("test_mode"):
        print("\n[PERINGATAN] test_mode aktif: artefak stage4 dihitung dari subset kecil. "
              "Kolom diversity pada report ini hanya indikatif.")

    master_data = []
    counts = Counter()

    for r in raw_records:
        rid = r.get("record_id")
        subtask = r.get(config["stratify_key"], "?")
        flags = r.get("flags", {}) or {}

        rule_rejected = any(flags.get(k) is True for k in RULE_BASED_HARD_GATE_KEYS)
        reject_reasons = [k for k in RULE_BASED_HARD_GATE_KEYS if flags.get(k) is True]
        soft_flags = [k for k in SOFT_FLAG_KEYS if flags.get(k) is True]

        s1 = stage1_map.get(rid, {})
        s2 = stage2_map.get(rid, {})
        s2c = stage2c_map.get(rid, {})

        ragas_f = _to_float(s2.get("ragas_faithfulness"))
        ragas_r = _to_float(s2.get("ragas_relevancy"))
        de_f = _to_float(s2c.get("deepeval_faithfulness"))
        de_ar = _to_float(s2c.get("deepeval_answer_relevancy"))
        de_cr = _to_float(s2c.get("deepeval_contextual_relevancy"))

        is_llm_judged = ragas_f is not None
        is_human_eval = rid in stage3_ids

        final_status = determine_final_decision(rule_rejected, is_llm_judged, ragas_f, de_f)
        counts[final_status] += 1

        if final_status == "REJECTED_RULE_BASED":
            reason = ", ".join(reject_reasons)
        elif final_status == "REJECTED_LOW_FAITHFULNESS":
            reason = f"low_faithfulness (ragas={ragas_f}, deepeval={de_f})"
        elif final_status == "UNVERIFIED_BY_LLM_JUDGE":
            reason = "lolos rule-based, tidak masuk sampel LLM-judge"
        else:
            reason = "verified_by_llm_judge"

        s4 = stage4_map.get(subtask, {})

        master_data.append({
            "record_id": rid,
            "subtask_type": subtask,
            "generation_focus": r.get("generation_focus", "-"),
            "question": r.get("instruction", ""),
            "answer": r.get("response", ""),
            "context": r.get("context", ""),
            "rule_based_rejected": rule_rejected,
            "soft_flags": ", ".join(soft_flags) if soft_flags else "-",
            "indobert_score": s1.get("indobert_score", "-"),
            "indobert_threshold_used": s1.get("threshold_used", "-"),
            "is_priority_sample_indobert": parse_bool(s1.get("flag_prefilter", False)),
            "is_llm_judged": is_llm_judged,
            "ragas_faithfulness": "-" if ragas_f is None else ragas_f,
            "ragas_relevancy": "-" if ragas_r is None else ragas_r,
            "deepeval_faithfulness": "-" if de_f is None else de_f,
            "deepeval_answer_relevancy": "-" if de_ar is None else de_ar,
            "deepeval_contextual_relevancy": "-" if de_cr is None else de_cr,
            "is_human_eval_sample": is_human_eval,
            "final_decision": final_status,
            "rejection_reason": reason,
            "stratum_level_distinct_2": s4.get("distinct_2", "-"),
            "stratum_level_count": s4.get("count", "-"),
        })

    df = pd.DataFrame(master_data)
    output_path = f"{out_dir}/master_qc_dataset_report{suffix}.csv"
    df.to_csv(output_path, index=False, encoding="utf-8")

    n_total = len(df)
    print(f"\nMaster QC Report disimpan ke: {output_path}")
    print(f"Total records: {n_total:,}\n")
    print(f"{'status':<28} {'jumlah':>9} {'persen':>8}")
    for status in ["REJECTED_RULE_BASED", "PASSED", "REJECTED_LOW_FAITHFULNESS",
                   "UNVERIFIED_BY_LLM_JUDGE"]:
        c = counts.get(status, 0)
        print(f"{status:<28} {c:>9,} {c/n_total*100:>7.1f}%")

    n_judged = int(df["is_llm_judged"].sum())
    n_human = int(df["is_human_eval_sample"].sum())
    n_soft = int((df["soft_flags"] != "-").sum())
    print(f"\ndinilai LLM-judge   : {n_judged:,} ({n_judged/n_total*100:.1f}%)")
    print(f"sampel human eval   : {n_human:,}")
    print(f"kena soft flag      : {n_soft:,} (tidak memengaruhi final_decision)")

    write_summary({
        "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "n_total": n_total,
        "final_decision_counts": dict(counts),
        "n_llm_judged": n_judged,
        "n_human_eval": n_human,
        "n_soft_flagged": n_soft,
        "hard_gate_keys": RULE_BASED_HARD_GATE_KEYS,
        "soft_flag_keys": SOFT_FLAG_KEYS,
        "llm_judge_threshold": LLM_JUDGE_FAITHFULNESS_THRESHOLD,
        "test_mode": config.get("test_mode", False),
    }, f"{out_dir}/master_qc_summary{suffix}.json")

    print("\n[CATATAN METODOLOGIS] Distribusi di atas tidak mencerminkan proporsi kualitas "
          "dataset untuk kategori UNVERIFIED_BY_LLM_JUDGE. Estimasi kualitas populasi hanya "
          "valid dihitung dari subset is_llm_judged=True, yang merupakan sampel stratified "
          "representatif — bukan dari seluruh populasi.")

    return df


df_master_result = generate_master_csv_report(CONFIG, raw_records)
df_master_result.head()

memuat artefak stage:
  stage1_prefilter.csv: 20,855 baris
  stage2_ragas.csv: 2,088 baris
  stage2c_deepeval.csv: 1,605 baris
  stage3_human_eval_template.csv: 99 baris
  stage4_stratum_coverage.csv: 9 baris

Master QC Report disimpan ke: /kaggle/working/qc/master_qc_dataset_report.csv
Total records: 20,855

status                          jumlah   persen
REJECTED_RULE_BASED                  0     0.0%
PASSED                           1,933     9.3%
REJECTED_LOW_FAITHFULNESS          156     0.7%
UNVERIFIED_BY_LLM_JUDGE         18,766    90.0%

dinilai LLM-judge   : 2,089 (10.0%)
sampel human eval   : 99
kena soft flag      : 0 (tidak memengaruhi final_decision)

[CATATAN METODOLOGIS] Distribusi di atas tidak mencerminkan proporsi kualitas dataset untuk kategori UNVERIFIED_BY_LLM_JUDGE. Estimasi kualitas populasi hanya valid dihitung dari subset is_llm_judged=True, yang merupakan sampel stratified representatif — bukan dari seluruh populasi.


,record_id,subtask_type,generation_focus,question,answer,context,rule_based_rejected,soft_flags,indobert_score,indobert_threshold_used,...,ragas_faithfulness,ragas_relevancy,deepeval_faithfulness,deepeval_answer_relevancy,deepeval_contextual_relevancy,is_human_eval_sample,final_decision,rejection_reason,stratum_level_distinct_2,stratum_level_count
0,30d661d4c3b0b0db,factual_retrieval,literal_reading,Berapa persen masyarakat yang pernah berurusan...,"Pada tahun 2022, persentase masyarakat yang me...",5.3 Membayar Melebihi Ketentuan Dalam upaya gl...,False,-,0.4706,0.2515,...,-,-,-,-,-,False,UNVERIFIED_BY_LLM_JUDGE,"lolos rule-based, tidak masuk sampel LLM-judge",0.2592,2690
1,1ea13f034163970d,source_navigation,literal_reading,Di bagian mana saya dapat menemukan pembahasan...,Pembahasan mengenai indikator SDGs 16.5.2 terd...,5.3 Membayar Melebihi Ketentuan Dalam upaya gl...,False,-,0.5525,0.2328,...,-,-,-,-,-,False,UNVERIFIED_BY_LLM_JUDGE,"lolos rule-based, tidak masuk sampel LLM-judge",0.2175,2215
2,dbf15de96152af24,factual_retrieval,literal_reading,Berapa persentase pelaku usaha yang pernah ber...,"Pada tahun 2020, tercatat bahwa 19,97 persen p...",5.3 Membayar Melebihi Ketentuan Dalam upaya gl...,False,-,0.5315,0.2515,...,-,-,-,-,-,False,UNVERIFIED_BY_LLM_JUDGE,"lolos rule-based, tidak masuk sampel LLM-judge",0.2592,2690
3,787a567728be1872,source_navigation,literal_reading,Di mana saya dapat menemukan grafik yang menun...,Grafik tersebut dapat ditemukan pada Gambar 5....,5.3 Membayar Melebihi Ketentuan Dalam upaya gl...,False,-,0.5369,0.2328,...,-,-,-,-,-,False,UNVERIFIED_BY_LLM_JUDGE,"lolos rule-based, tidak masuk sampel LLM-judge",0.2175,2215
4,64ad4e0f6f5794ad,definition_clarification,concept_understanding,Apa yang dimaksud dengan Indikator SDGs 16.5.1?,Indikator SDGs 16.5.1 mengukur tingkat penyuap...,5.3 Membayar Melebihi Ketentuan Dalam upaya gl...,False,-,0.3901,0.274,...,-,-,-,-,-,False,UNVERIFIED_BY_LLM_JUDGE,"lolos rule-based, tidak masuk sampel LLM-judge",0.3275,2085


## 7. Ringkasan Estimasi Kualitas dari Sampel Terverifikasi

In [15]:
def summarize_verified_subset(df, config):
    judged = df[df["is_llm_judged"] == True].copy()
    if judged.empty:
        print("belum ada record yang dinilai LLM-judge.")
        return None

    metrics = ["ragas_faithfulness", "ragas_relevancy",
               "deepeval_faithfulness", "deepeval_answer_relevancy",
               "deepeval_contextual_relevancy"]

    print(f"subset terverifikasi: {len(judged):,} record "
          f"({len(judged)/len(df)*100:.1f}% dari populasi)\n")

    rows = []
    for m in metrics:
        vals = pd.to_numeric(judged[m], errors="coerce").dropna()
        if vals.empty:
            continue
        mean = vals.mean()
        std = vals.std(ddof=1) if len(vals) > 1 else 0.0
        se = std / (len(vals) ** 0.5) if len(vals) else 0.0
        rows.append({
            "metric": m,
            "n": len(vals),
            "mean": round(mean, 4),
            "std": round(std, 4),
            "se": round(se, 5),
            "ci95_low": round(mean - 1.96 * se, 4),
            "ci95_high": round(mean + 1.96 * se, 4),
            "pct_below_0.5": round((vals < 0.5).mean() * 100, 2),
        })

    est = pd.DataFrame(rows)
    print(est.to_string(index=False))

    suffix = "_test" if config.get("test_mode") else ""
    path = f"{config['output_dir']}/stage5_verified_subset_estimates{suffix}.csv"
    est.to_csv(path, index=False, encoding="utf-8")
    print(f"\ndisimpan ke: {path}")

    print("\nper subtask_type (mean ragas_faithfulness):")
    judged["_rf"] = pd.to_numeric(judged["ragas_faithfulness"], errors="coerce")
    by_st = judged.groupby("subtask_type")["_rf"].agg(["count", "mean"]).sort_values("mean")
    for st, row in by_st.iterrows():
        print(f"  {st:<28} n={int(row['count']):>4}  mean={row['mean']:.3f}")

    print("\nCI 95% dihitung dengan asumsi sampel acak dan distribusi mendekati normal. "
          "Karena sampling bersifat stratified dengan proporsi flag/general yang diatur, "
          "interval ini adalah aproksimasi — bukan estimasi berbobot penuh.")
    return est


estimates = summarize_verified_subset(df_master_result, CONFIG)

subset terverifikasi: 2,089 record (10.0% dari populasi)

                       metric    n   mean    std      se  ci95_low  ci95_high  pct_below_0.5
           ragas_faithfulness 2089 0.7326 0.3580 0.00783    0.7172     0.7479          20.63
              ragas_relevancy 2089 0.6458 0.1873 0.00410    0.6377     0.6538          15.46
        deepeval_faithfulness 1606 0.8512 0.2643 0.00660    0.8383     0.8642           7.53
    deepeval_answer_relevancy 1606 0.9424 0.1566 0.00391    0.9347     0.9500           2.05
deepeval_contextual_relevancy   78 0.2744 0.3413 0.03864    0.1987     0.3502          65.38

disimpan ke: /kaggle/working/qc/stage5_verified_subset_estimates.csv

per subtask_type (mean ragas_faithfulness):
  causal_reasoning             n= 255  mean=0.628
  multi_source_synthesis       n= 255  mean=0.695
  factual_retrieval            n= 269  mean=0.714
  definition_clarification     n= 210  mean=0.730
  trend_interpretation         n= 253  mean=0.739
  methodology_expla

## 8. List output

In [16]:
def list_outputs(config):
    d = Path(config["output_dir"])
    files = sorted([f for f in d.iterdir() if f.is_file()])
    total = sum(f.stat().st_size for f in files)
    print(f"{len(files)} berkas di {d} (total {total/1024:.1f} KB)\n")
    for f in files:
        print(f"  {f.name:<52} {f.stat().st_size:>10,} bytes")
    print("\nSemua berkas di atas akan tersimpan sebagai output versi notebook ini "
          "setelah Save Version, dan dapat dipakai sebagai input notebook berikutnya.")


list_outputs(CONFIG)

19 berkas di /kaggle/working/qc (total 153851.8 KB)

  environment_snapshot.json                                   698 bytes
  environment_snapshot_tahap4.json                            561 bytes
  master_qc_dataset_report.csv                         153,575,463 bytes
  master_qc_summary.json                                      539 bytes
  stage1_prefilter.csv                                  1,610,631 bytes
  stage1_summary.json                                         608 bytes
  stage2_ragas.csv                                        197,219 bytes
  stage2_ragas_checkpoint.csv                              88,572 bytes
  stage2c_deepeval.csv                                    729,791 bytes
  stage2c_deepeval_checkpoint.csv                         666,842 bytes
  stage3_human_eval_template.csv                          669,462 bytes
  stage3_summary.json                                         518 bytes
  stage4_diversity_summary.csv                                250 bytes
  stage4_e